# 01a · Head-level attribution

**Question:** which attention heads (and MLP layers) carry the sarcasm information, and in which region of the prompt?

**Method:** attribution patching for every node at once (one forward + backward pass per pair):

$$\widehat{R}_{h,\,\text{region}} = \frac{\sum_{p \in \text{region}} (z^{\text{clean}}_{h,p} - z^{\text{corr}}_{h,p}) \cdot \partial d / \partial z_{h,p}}{d(S) - d(\text{corrupted})}$$

an estimate of the recovery $R$ we would get by patching head $h$ only in that region.

**Validation:** for a subset of pairs we also run **real** activation patching of every head in every region,
and compare it with the estimate. If they agree, the gradients can be trusted (also on the 4-bit 7B model)
and Phase 1b (EAP-IG on edges) can build on them.

**Runtime:** T4 GPU. Start with `MODEL = DEV_MODEL` (3B, ~10 min); when everything works, set `MODEL = MAIN_MODEL` (7B, ~30 min).
Results are saved to Google Drive, so a rerun continues where it stopped.

## Setup

In [ ]:
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT)
%pip install -q -U accelerate bitsandbytes

In [ ]:
import os, numpy as np, pandas as pd, torch, time
import matplotlib.pyplot as plt
from IPython.display import display
from src.model import load_model, model_info, DEV_MODEL, MAIN_MODEL
from src.prompts import load_dataset, make_pairs, REGIONS
from src.patching import score_pairs, keep_correct, mean_and_se
from src.attribution import attribution_over_pairs, actual_head_R, top_heads
from src.utils import mount_drive, results_dir, cached, set_seed

# ---------------- settings ----------------
MODEL = DEV_MODEL          # change to MAIN_MODEL for the final run
N_VALIDATE = 8             # pairs per experiment for the real-patching check
TOP_K = 20
# ------------------------------------------
TAG = MODEL.split("/")[-1]
SUB = f"01a_head_attribution/{TAG}"
mount_drive()
print("results go to:", results_dir(SUB))
set_seed(0)

In [ ]:
t0 = time.time()
model, tok = load_model(MODEL)
INFO = model_info(model)
print(MODEL, INFO, f"loaded in {(time.time() - t0) / 60:.1f} min")

## Pairs: all token-matched discovery pairs that the model answers correctly

In [ ]:
df = load_dataset()
PAIRS = {}
for e in ["situation", "phrase"]:
    ps = score_pairs(model, tok, make_pairs(df, tok, e))
    PAIRS[e] = keep_correct(ps)
    print(f"{e:9s}: {len(PAIRS[e])} of {len(ps)} pairs answered correctly")

## Attribution for every head and MLP (cached)

In [ ]:
t0 = time.time()
ATTR = {e: cached(f"attribution_{e}", lambda e=e: attribution_over_pairs(model, tok, PAIRS[e]), sub=SUB)
        for e in PAIRS}
print(f"done in {(time.time() - t0) / 60:.1f} min")
for e, a in ATTR.items():
    print(e, "heads", a["heads"].shape, "mlp", a["mlp"].shape)

### Heads: estimated $R$ per layer and head, for each region

Blue = patching this head alone moves the answer towards sarcasm; red = it pushes the other way.
Most heads should be white: a sparse picture is what H1 predicts.

In [ ]:
for e, a in ATTR.items():
    m = a["heads"].mean(0)                                  # [L, H, R]
    lim = max(0.05, np.abs(m).max())
    fig, axes = plt.subplots(1, len(REGIONS), figsize=(4.2 * len(REGIONS), 4.6), sharey=True)
    for j, (ax, r) in enumerate(zip(axes, REGIONS)):
        im = ax.imshow(m[:, :, j], aspect="auto", origin="lower", cmap="RdBu", vmin=-lim, vmax=lim)
        ax.set_title(r); ax.set_xlabel("head")
    axes[0].set_ylabel("layer")
    fig.colorbar(im, ax=axes, shrink=0.85, label="estimated R (mean over pairs)")
    fig.suptitle(f"{TAG}: {e} swap, {a['heads'].shape[0]} pairs")
    plt.savefig(os.path.join(results_dir(SUB), f"heads_{e}.png"), dpi=150, bbox_inches="tight")
    plt.show()

### MLP layers: estimated $R$ per layer and region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6), sharey=True)
for ax, (e, a) in zip(axes, ATTR.items()):
    m = a["mlp"].mean(0)
    lim = max(0.05, np.abs(m).max())
    im = ax.imshow(m, aspect="auto", origin="lower", cmap="RdBu", vmin=-lim, vmax=lim)
    ax.set_xticks(range(len(REGIONS))); ax.set_xticklabels(REGIONS, rotation=30)
    ax.set_title(f"{e} swap"); plt.colorbar(im, ax=ax, shrink=0.85)
axes[0].set_ylabel("layer")
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "mlp.png"), dpi=150); plt.show()

### Top heads

In [ ]:
TOP = {}
for e, a in ATTR.items():
    m, se = mean_and_se(a["heads"])
    rows = top_heads(m, TOP_K)
    for r in rows:
        r["se"] = float(se[r["layer"], r["head"], REGIONS.index(r["region"])])
    TOP[e] = pd.DataFrame(rows)
    print(f"\n{e} swap: top {TOP_K} (layer, head, region) by estimated R")
    display(TOP[e].round(3))

## Validation: estimate vs real activation patching

For `N_VALIDATE` pairs per experiment we patch **every head in every region for real** and compare.
What we want to see: points close to the diagonal, a high correlation, and the same heads at the top.

In [ ]:
t0 = time.time()
ACT = {e: cached(f"actual_{e}_{N_VALIDATE}pairs",
                 lambda e=e: np.stack([actual_head_R(model, tok, p) for p in PAIRS[e][:N_VALIDATE]]), sub=SUB)
       for e in PAIRS}
print(f"done in {(time.time() - t0) / 60:.1f} min")

VAL = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, e in zip(axes, PAIRS):
    est = ATTR[e]["heads"][:N_VALIDATE]                     # same pairs as ACT
    act = ACT[e]
    x, y = est.ravel(), act.ravel()
    r = np.corrcoef(x, y)[0, 1]
    me, ma = est.mean(0), act.mean(0)
    top_est = set(np.argsort(me, axis=None)[::-1][:TOP_K])
    top_act = set(np.argsort(ma, axis=None)[::-1][:TOP_K])
    overlap = len(top_est & top_act) / TOP_K
    VAL[e] = dict(pearson=r, top_overlap=overlap)
    lim = max(np.abs(x).max(), np.abs(y).max()) * 1.05
    ax.scatter(x, y, s=4, alpha=0.3)
    ax.plot([-lim, lim], [-lim, lim], c="gray", lw=0.8, ls="--")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel("estimated R (attribution)"); ax.set_ylabel("real R (activation patching)")
    ax.set_title(f"{e} swap: r = {r:.2f}, top-{TOP_K} overlap = {overlap:.0%}")
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "validation.png"), dpi=150); plt.show()
for e, v in VAL.items():
    print(f"{e:9s}  Pearson r = {v['pearson']:.3f}   top-{TOP_K} overlap = {v['top_overlap']:.0%}")

**How to read it.** $r > 0.8$ and an overlap of at least 60–70% mean the gradients point to the right heads and Phase 1b can use them.
Points far from the diagonal are heads where the linear estimate fails (often in saturated layers); a few are expected,
and these are exactly what EAP-IG's integrated gradients are meant to fix.

## Save a table of all heads (for the report and for Phase 1b)

In [ ]:
rows = []
for e, a in ATTR.items():
    m, se = mean_and_se(a["heads"])
    act_m = ACT[e].mean(0)
    for l in range(m.shape[0]):
        for h in range(m.shape[1]):
            for j, r in enumerate(REGIONS):
                rows.append(dict(experiment=e, layer=l, head=h, region=r, est_R=m[l, h, j], est_se=se[l, h, j],
                                 real_R=act_m[l, h, j]))
table = pd.DataFrame(rows)
path = os.path.join(results_dir(SUB), "head_scores.csv")
table.to_csv(path, index=False)
pd.DataFrame(VAL).T.to_csv(os.path.join(results_dir(SUB), "validation.csv"))
print("saved", path, len(table), "rows")